# Member 4 - Preprocessing Pipeline

**Project:** Hotel Booking Cancellation Prediction  
**Scope:** Categorical encoding, numerical preprocessing, missing-value imputation, stratified train/test preparation, and a reusable preprocessing object.

> This notebook intentionally stops after preprocessing. It does not train, tune, compare, or evaluate any machine-learning model.

## 1. Continue from Members 1-3

Member 1 completed exploratory analysis. Member 2 removed exact duplicates and invalid records. Member 3 removed confirmed post-outcome leakage and created eight domain features. To keep this notebook reproducible, the established Member 2 and Member 3 functions are called below to recreate their final `df_engineered` handoff from the raw CSV. Their cleaning and feature-selection decisions are not redefined here.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from scipy import sparse

# Locate the repository whether Jupyter starts in the repo root or this notebook folder.
repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in repo_candidates if (path / 'data/raw/hotel_bookings.csv').exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate data/raw/hotel_bookings.csv from the current directory.')

src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from feature_engineering.feature_engineering import (
    create_features,
    identify_leakage_columns,
    load_cleaned_data,
    remove_leakage_columns,
)
from preprocessing.preprocessing import (
    IDENTIFIER_CATEGORICAL_COLUMNS,
    KNOWN_LEAKAGE_COLUMNS,
    build_preprocessor,
    build_verification_report,
    identify_feature_types,
    matrix_invalid_counts,
    prepare_features_target,
    processed_preview,
    split_data,
    target_distribution_table,
)

In [2]:
# Reproduce the exact handoff produced by Members 2 and 3.
DATA_PATH = REPO_ROOT / 'data/raw/hotel_bookings.csv'
df_clean = load_cleaned_data(DATA_PATH)
member3_leakage_columns = identify_leakage_columns()['confirmed_leakage']
df_no_leakage = remove_leakage_columns(df_clean, member3_leakage_columns)
df_engineered = create_features(df_no_leakage)

print(f'Member 3 handoff recreated: {df_engineered.shape[0]:,} rows x {df_engineered.shape[1]} columns')
print('Input dataframe for Member 4: df_engineered')

Member 3 handoff recreated: 87,228 rows x 38 columns
Input dataframe for Member 4: df_engineered


## 2. Prepare Predictors and Target

The target is separated because `is_canceled` is the outcome to predict, not an input feature. The safety function also excludes `reservation_status` and `reservation_status_date` if they ever reappear, and emits a warning so a broken Member 3 handoff cannot silently leak outcome information into preprocessing.

In [3]:
X, y, defensively_removed_leakage = prepare_features_target(
    df_engineered,
    target_col='is_canceled',
)
numerical_features, categorical_features = identify_feature_types(X)

# agent and company are numeric codes, not measured quantities. Treating them as
# nominal categories avoids imposing a false numerical order during scaling.
identifier_overrides_used = [
    column for column in IDENTIFIER_CATEGORICAL_COLUMNS if column in X.columns
]

input_summary = pd.DataFrame({
    'Measure': [
        'Rows', 'Predictor columns', 'X shape', 'y shape',
        'Numerical features', 'Categorical features'
    ],
    'Value': [
        f'{len(X):,}', len(X.columns), str(X.shape), str(y.shape),
        len(numerical_features), len(categorical_features)
    ],
})
display(input_summary)

target_summary = pd.DataFrame({
    'Count': y.value_counts().sort_index(),
    'Percentage (%)': (y.value_counts(normalize=True).sort_index() * 100).round(2),
})
target_summary.index = ['Not canceled (0)', 'Canceled (1)']
display(target_summary)

print(f'Numerical feature names ({len(numerical_features)}):')
print(numerical_features)
print(f'\nCategorical feature names ({len(categorical_features)}):')
print(categorical_features)
print(f'\nIdentifier-code categorical overrides: {identifier_overrides_used}')
leakage_message = defensively_removed_leakage or 'None (already removed by Member 3)'
print(f'Defensively removed leakage columns: {leakage_message}')

remaining_missing = X.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print('\nRemaining missing values before pipeline imputation:')
display(remaining_missing.rename('Missing count').to_frame() if not remaining_missing.empty else pd.DataFrame({'Missing count': []}))

,Measure,Value
0,Rows,"87,228"
1,Predictor columns,37
2,X shape,"(87228, 37)"
3,y shape,"(87228,)"
4,Numerical features,25
5,Categorical features,12


,Count,Percentage (%)
Not canceled (0),63220,72.48
Canceled (1),24008,27.52


Numerical feature names (25):
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'total_stay', 'is_family', 'room_changed', 'has_special_requests', 'has_previous_cancellations', 'has_previous_bookings', 'is_weekend_only']

Categorical feature names (12):
['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'agent', 'company']

Identifier-code categorical overrides: ['agent', 'company']
Defensively removed leakage columns: None (already removed by Member 3)

Remaining missing values before pipeline imputation:


,Missing count
company,81991
agent,12141
country,447
children,4


## 3. Stratified Train/Test Split

The 80/20 split occurs **before any learned transformation**. This prevents test-set values from influencing imputation statistics, scaling parameters, or one-hot categories. `stratify=y` keeps the cancellation proportion approximately equal in the full, training, and test sets, which makes both partitions representative of the target distribution. `random_state=42` makes the split reproducible.

In [4]:
X_train, X_test, y_train, y_test = split_data(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

split_shapes = pd.DataFrame({
    'Object': ['X_train', 'X_test', 'y_train', 'y_test'],
    'Shape': [str(X_train.shape), str(X_test.shape), str(y_train.shape), str(y_test.shape)],
})
display(split_shapes)

class_distribution = target_distribution_table(y, y_train, y_test)
display(class_distribution)

,Object,Shape
0,X_train,"(69782, 37)"
1,X_test,"(17446, 37)"
2,y_train,"(69782,)"
3,y_test,"(17446,)"


,Dataset,Class,Meaning,Count,Percentage (%)
0,Full,0,Not canceled,63220,72.48
1,Full,1,Canceled,24008,27.52
2,Train,0,Not canceled,50576,72.48
3,Train,1,Canceled,19206,27.52
4,Test,0,Not canceled,12644,72.48
5,Test,1,Canceled,4802,27.52


## 4. Reusable Numerical and Categorical Pipelines

The numerical pipeline uses median imputation because the numerical variables include skewed distributions and extreme but valid observations. `StandardScaler` then places numerical variables on comparable scales, keeping the data suitable for later scale-sensitive algorithms such as Logistic Regression, SVM, and KNN.

The categorical pipeline uses most-frequent imputation for remaining missing categories and `OneHotEncoder` because the categories are nominal and have no meaningful order. `handle_unknown='ignore'` allows test or future records containing a category unseen during training to be transformed without an error. A `ColumnTransformer` applies each pipeline only to its corresponding columns and drops no approved predictor accidentally.

In [5]:
preprocessor = build_preprocessor(numerical_features, categorical_features)

# Keep named references for inspection and later reuse.
numerical_pipeline = preprocessor.transformers[0][1]
categorical_pipeline = preprocessor.transformers[1][1]

print('Numerical pipeline:')
display(numerical_pipeline)
print('Categorical pipeline:')
display(categorical_pipeline)
print('Combined preprocessor:')
display(preprocessor)

Numerical pipeline:


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

Categorical pipeline:


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto acc

Combined preprocessor:


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## 5. Fit on Training Data Only

`fit_transform` is called only on `X_train`. The already-fitted object then calls `transform` on `X_test`; the test set is never fitted. Therefore every learned median, mode, mean, standard deviation, and category vocabulary comes only from the training partition.

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

processed_feature_names = preprocessor.get_feature_names_out()

print(f'Original predictor count: {X.shape[1]}')
print(f'Processed feature count:  {len(processed_feature_names)}')
print(f'X_train_processed shape:  {X_train_processed.shape}')
print(f'X_test_processed shape:   {X_test_processed.shape}')
output_storage_type = 'sparse' if sparse.issparse(X_train_processed) else 'dense'
print(f'Output storage type:       {output_storage_type}')
print('\nFirst 25 processed feature names:')
print(processed_feature_names[:25].tolist())

Original predictor count: 37
Processed feature count:  899
X_train_processed shape:  (69782, 899)
X_test_processed shape:   (17446, 899)
Output storage type:       sparse

First 25 processed feature names:
['num__lead_time', 'num__arrival_date_year', 'num__arrival_date_week_number', 'num__arrival_date_day_of_month', 'num__stays_in_weekend_nights', 'num__stays_in_week_nights', 'num__adults', 'num__children', 'num__babies', 'num__is_repeated_guest', 'num__previous_cancellations', 'num__previous_bookings_not_canceled', 'num__booking_changes', 'num__days_in_waiting_list', 'num__adr', 'num__required_car_parking_spaces', 'num__total_of_special_requests', 'num__total_guests', 'num__total_stay', 'num__is_family', 'num__room_changed', 'num__has_special_requests', 'num__has_previous_cancellations', 'num__has_previous_bookings', 'num__is_weekend_only']


## 6. Preprocessing Verification

The checks below confirm matching transformed columns, finite numerical output, unchanged targets, correct split proportions, preserved class balance, and the absence of the target and known leakage fields. Sparse matrices are checked through their stored values, so the complete dataset is never converted to a large dense array.

In [7]:
verification_report = build_verification_report(
    X=X,
    y=y,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    X_train_processed=X_train_processed,
    X_test_processed=X_test_processed,
    processed_feature_names=processed_feature_names,
)
display(verification_report)

train_invalid_counts = matrix_invalid_counts(X_train_processed)
test_invalid_counts = matrix_invalid_counts(X_test_processed)
invalid_value_summary = pd.DataFrame([
    {'Dataset': 'Training', **train_invalid_counts},
    {'Dataset': 'Testing', **test_invalid_counts},
])
display(invalid_value_summary)

processed_names_as_text = [str(name) for name in processed_feature_names]
assert 'is_canceled' not in X.columns
assert not any(column in X.columns for column in KNOWN_LEAKAGE_COLUMNS)
assert 'is_canceled' not in numerical_features
assert 'is_canceled' not in categorical_features
assert not any(column in numerical_features for column in KNOWN_LEAKAGE_COLUMNS)
assert not any(column in categorical_features for column in KNOWN_LEAKAGE_COLUMNS)
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert train_invalid_counts['NaN count'] == 0
assert test_invalid_counts['NaN count'] == 0
assert train_invalid_counts['Infinity count'] == 0
assert test_invalid_counts['Infinity count'] == 0
assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})
assert not any('is_canceled' in name for name in processed_names_as_text)
assert not any(
    leakage_column in name
    for leakage_column in KNOWN_LEAKAGE_COLUMNS
    for name in processed_names_as_text
)
assert verification_report['Status'].eq('PASS').all(), 'One or more preprocessing checks failed.'
print('ALL MEMBER 4 PREPROCESSING CHECKS PASSED')

,Verification check,Status
0,Processed matrices created,PASS
1,Training feature/target row counts match,PASS
2,Test feature/target row counts match,PASS
3,Train/test processed column counts match,PASS
4,Processed feature-name count matches,PASS
5,Training processed data has no NaN values,PASS
6,Test processed data has no NaN values,PASS
7,Training processed data has no infinite values,PASS
8,Test processed data has no infinite values,PASS
9,Training target remains binary,PASS


,Dataset,NaN count,Infinity count
0,Training,0,0
1,Testing,0,0


ALL MEMBER 4 PREPROCESSING CHECKS PASSED


## 7. Small Processed Training Preview

Only the first five rows and first twelve transformed columns are converted for display. The full processed matrix remains in its memory-efficient output format.

In [8]:
display(
    processed_preview(
        X_train_processed,
        processed_feature_names,
        rows=5,
        columns=12,
    )
)

,num__lead_time,num__arrival_date_year,num__arrival_date_week_number,num__arrival_date_day_of_month,num__stays_in_weekend_nights,num__stays_in_week_nights,num__adults,num__children,num__babies,num__is_repeated_guest,num__previous_cancellations,num__previous_bookings_not_canceled
0,0.527026,1.151130,-1.595838,-1.339449,4.865554,6.085823,-1.473609,-0.305543,-0.093378,-0.199887,-0.080144,-0.10671
1,0.154351,1.151130,-1.888113,-1.452654,-0.977381,-0.305953,0.203600,-0.305543,-0.093378,-0.199887,-0.080144,-0.10671
2,-0.893797,-1.765492,1.838399,1.037874,-0.977381,-0.797628,0.203600,-0.305543,-0.093378,-0.199887,-0.080144,-0.10671
3,1.109331,1.151130,-1.011287,1.490697,-0.977381,0.677397,0.203600,-0.305543,-0.093378,-0.199887,-0.080144,-0.10671
4,-0.742398,-1.765492,1.838399,1.037874,0.970264,0.677397,-1.473609,-0.305543,-0.093378,-0.199887,2.464266,-0.10671


## Preventing Data Leakage During Preprocessing

- The train/test split happens before fitting any learned transformation.
- Numerical imputation medians are learned only from `X_train`.
- Categorical imputation modes are learned only from `X_train`.
- `StandardScaler` means and standard deviations are learned only from `X_train`.
- `OneHotEncoder` category vocabularies are learned only from `X_train`.
- `X_test` is transformed using the already-fitted training transformations; it is never passed to `fit` or `fit_transform`.
- `is_canceled` is separated before type detection, splitting, and transformation, so it can never become a predictor.
- Member 3's `reservation_status` and `reservation_status_date` leakage fields are excluded before preprocessing. A defensive Member 4 check also removes and reports them if they unexpectedly reappear.

Together, these controls ensure the held-out test data contributes no information to the preprocessing parameters. The resulting test set remains a valid unseen partition for a later project stage.

## 8. Handoff for the Later Modeling Stage

The following reusable objects are now available in memory:

- `preprocessor`
- `X_train`, `X_test`, `y_train`, `y_test`
- `X_train_processed`, `X_test_processed`
- `processed_feature_names`

No fitted binary artifact is committed at Progress Evaluation 1. In a later approved stage, the same fitted `preprocessor` can be placed before an estimator in a scikit-learn `Pipeline` or saved together with that complete model workflow. No model is trained here.